# Import proteins from a comma-separated values (CSV) file

`ProteinCsvImport.start` adds a CSV of proteins of any size to your project. It uploads the
file, starts the import and returns straight away. `wait()` blocks until the import has
finished, then returns every protein in the project as `Protein` objects, without
downloading their structures.

In [ ]:
from deeporigin import projects
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient.from_disk()
projects.create("python-client-dev", client=client)

## Start the import

Each column header is a platform protein field (`protein_name`, `fasta_sequence`, `pdb_id`,
`file_path`, ...). Here the CSV has names and sequences only.

In [ ]:
import tempfile
import time
from pathlib import Path

from deeporigin.drug_discovery import ProteinCsvImport

suffix = time.strftime("%Y%m%dT%H%M%S")
rows = {
    f"ubiquitin-{suffix}": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
    f"insulin-b-{suffix}": "FVNQHLCGSHLVEALYLVCGERGFFYTPKT",
    f"glucagon-{suffix}": "HSQGTFTSDYSKYLDSRRAQDFVQWLMNT",
}
local_csv = Path(tempfile.mkdtemp()) / "proteins.csv"
local_csv.write_text(
    "protein_name,fasta_sequence\n"
    + "".join(f"{name},{seq}\n" for name, seq in rows.items())
)

start = time.monotonic()
job = ProteinCsvImport.start(local_csv, client=client)
print(f"returned after {time.monotonic() - start:.1f}s")
job.execution_id, job.project_id

## Wait for the proteins

`wait()` raises `DeepOriginException` if the import fails and `TimeoutError` if it has not
finished within `timeout` seconds (one hour by default).

A row whose sequence is already in the project reuses that protein, which keeps its existing
name, so the rows are checked by sequence.

In [ ]:
start = time.monotonic()
proteins = job.wait()
print(f"returned after {time.monotonic() - start:.1f}s")


def proteins_with_sequence(sequence):
    return client.entities.search_proteins(project_id=job.project_id, sequence=sequence)["data"]


returned_ids = {p.id for p in proteins}
{name: [r["protein_name"] for r in proteins_with_sequence(seq)] for name, seq in rows.items()}

In [ ]:
for name, seq in rows.items():
    matches = proteins_with_sequence(seq)
    if len(matches) != 1 or matches[0]["id"] not in returned_ids:
        raise RuntimeError(f"Expected one protein with the sequence of {name}, found {len(matches)}")

## Import again

A protein whose sequence is already in the project is reused rather than imported twice, so
importing the same CSV again adds no duplicates.

In [ ]:
ProteinCsvImport.start(local_csv, client=client).wait()
{name: len(proteins_with_sequence(seq)) for name, seq in rows.items()}

## Attach structure files

To attach structures, put the Protein Data Bank (PDB) files in a zip, name each row's file in `file_path`,
and pass the zip as `protein_zip_path`. Structures are not downloaded by `wait()`; call
`download()` on a protein when you need its coordinates.

A structure that is already in the project is reused, and keeps its existing name.

In [ ]:
import zipfile

from deeporigin.drug_discovery import BRD_DATA_DIR

structures_zip = local_csv.parent / "structures.zip"
with zipfile.ZipFile(structures_zip, "w") as archive:
    archive.write(BRD_DATA_DIR / "brd.pdb", arcname="brd.pdb")

brd_name = f"brd4-{suffix}"
structures_csv = local_csv.parent / "proteins-with-structures.csv"
structures_csv.write_text(f"protein_name,file_path\n{brd_name},brd.pdb\n")

proteins = ProteinCsvImport.start(
    structures_csv, protein_zip_path=structures_zip, client=client
).wait()
[(p.name, p.id, p.remote_path) for p in proteins if p.remote_path]